# Notebook 07 — Iterative Burst Detector: Kill-Stage Diagnostic

Visualises each stage of the detection pipeline using `IterativeBurstTrace`.
Events are killed at four stages:

1. **Iteration trimming/merging** — candidates below the composite threshold during refinement.
2. **Participation floor** — burstlets with peak active-unit fraction below `participation_floor`.
3. **BMI / LLR gate** — burstlets with `llr_aggregate < config.min_burst_modulation` (default `0.1`).
4. **GMM event clustering** — 6-component GMM on 6D event features; components scoring < 0 are discarded.

The second half of this notebook (sections A–G) provides a deep-dive into the
per-bin LDA iteration geometry on any chosen recording.
The silence-excision + sign-pinning fix implemented in `iterative_burst_detector.py`
resolved the original `cx138_44_02` failure (Fisher direction flipping to `w_PFR = −0.81`;
sections F/G document what the 3-regime data structure looked like before the fix).

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

repo_root = Path.cwd().resolve()
if (repo_root / 'pipeline_tasks').is_dir() is False and (repo_root.parent / 'pipeline_tasks').is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline_tasks.analysis import (
    IterativeBurstConfig,
    IterativeBurstTrace,
    compute_iterative_bursts,
)


## Configuration

Edit the cell below before running. All other cells read from these variables.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                     CONFIGURATION                           ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Recording to diagnose (for LDA deep-dive sections A–G) ───────────────────
SOURCE_NAME = 'cx138_44_02'  # 'cx138_36_10' | 'cx138_36_11' | 'cx138_44_02'

# ── Detector config ──────────────────────────────────────────────────────────
config = IterativeBurstConfig()

# ── LDA-iteration visualisation controls (sections C / D) ────────────────────
TRACE_TO_PLOT  = 'default'    # 'default' (stock config) | 'no_gate' (BMI disabled)
PLOT_ALL_ITERS = False         # True → one PCA panel per iteration

# ── Output ───────────────────────────────────────────────────────────────────
PLOTLY_OFFLINE   = False  # True → embed Plotly JS in HTML (~3 MB); False → load from CDN
FIGURE_OUTPUT_DIR = repo_root / 'output' / 'nb07_diagnostic_figures'
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Figures will be saved to: {FIGURE_OUTPUT_DIR}')


## Load a real recording

Loader helpers copied from notebook 06 so the two notebooks see the same data.

In [ ]:
def is_kilosort_dir(path: Path) -> bool:
    return path.is_dir() and all((path / fn).exists() for fn in ('spike_times.npy', 'spike_clusters.npy', 'params.py'))

def discover_real_spike_sources(root: Path | None) -> list[Path]:
    if root is None or not root.exists():
        return []
    if root.is_file() or is_kilosort_dir(root):
        return [root]
    curated = sorted(root.rglob('curated_spike_times.npy'))
    ks_dirs = sorted({p.parent for p in root.rglob('spike_times.npy') if is_kilosort_dir(p.parent)})
    curated_parents = {p.parent for p in curated}
    sources = curated + [p for p in ks_dirs if p not in curated_parents]
    return sorted(set(sources), key=lambda p: str(p))

def _read_kilosort_sample_rate(params_path: Path) -> float:
    values: dict[str, object] = {}
    exec(params_path.read_text(), {}, values)
    for key in ('sample_rate', 'fs', 'sampling_rate'):
        if key in values:
            return float(values[key])
    raise ValueError(f'Could not find sample_rate/fs/sampling_rate in {params_path}')

def _read_kilosort_keep_clusters(ks_dir: Path, labels: set[str] | None) -> set[int] | None:
    if labels is None:
        return None
    for filename in ('cluster_group.tsv', 'cluster_KSLabel.tsv'):
        label_path = ks_dir / filename
        if not label_path.exists():
            continue
        table = pd.read_csv(label_path, sep='\t')
        if 'cluster_id' not in table.columns:
            continue
        label_col = next((c for c in ('group', 'KSLabel', 'label') if c in table.columns), None)
        if label_col is None:
            continue
        keep = table[table[label_col].astype(str).str.lower().isin({l.lower() for l in labels})]
        return set(keep['cluster_id'].astype(int).tolist())
    return None

def load_kilosort_spike_times(ks_dir: Path, labels: set[str] | None = REAL_KILOSORT_LABELS) -> dict[str, np.ndarray]:
    ks_dir = Path(ks_dir)
    sample_rate = _read_kilosort_sample_rate(ks_dir / 'params.py')
    spike_samples = np.load(ks_dir / 'spike_times.npy').reshape(-1).astype(float)
    spike_clusters = np.load(ks_dir / 'spike_clusters.npy').reshape(-1).astype(int)
    keep_clusters = _read_kilosort_keep_clusters(ks_dir, labels)
    out: dict[str, np.ndarray] = {}
    for cid in np.unique(spike_clusters):
        if keep_clusters is not None and int(cid) not in keep_clusters:
            continue
        times = spike_samples[spike_clusters == cid] / sample_rate
        times = times[np.isfinite(times) & (times >= 0.0)]
        if times.size:
            out[f'cluster_{int(cid):03d}'] = np.sort(times.astype(float))
    if not out:
        raise ValueError(f'No clusters found in {ks_dir}')
    return out


In [ ]:
REAL_DATA_ROOT       = repo_root / 'data' / 'real_sorted_data'
REAL_KILOSORT_LABELS = {'good'}

sources = discover_real_spike_sources(REAL_DATA_ROOT)
print(f'discovered {len(sources)} real spike source(s):')
for s in sources:
    print('  ', s)
assert sources, f'no real recordings found under {REAL_DATA_ROOT}'


In [ ]:
def save_html(fig: go.Figure, filename: str) -> Path:
    path = FIGURE_OUTPUT_DIR / filename
    plotlyjs = True if PLOTLY_OFFLINE else 'cdn'
    fig.write_html(str(path), include_plotlyjs=plotlyjs)
    print(f'Saved: {path}')
    return path


In [ ]:
# Run the detector on every discovered recording and cache results + traces.
all_spike_times: dict[str, dict] = {}
all_traces: dict[str, IterativeBurstTrace] = {}
all_results: dict = {}

for _source in sources:
    _name = _source.name if _source.is_dir() else _source.parent.name
    _st = load_kilosort_spike_times(_source)
    _tr = IterativeBurstTrace()
    _res = compute_iterative_bursts(_st, config=config, debug=True, trace=_tr)
    all_spike_times[_name] = _st
    all_traces[_name] = _tr
    all_results[_name] = _res
    print(f'{_name}: {len(_res.burstlets)} burstlets  '
          f'{len(_res.network_bursts)} net bursts')


## Kill-stage attribution at a glance

How many candidates survive each stage of the pipeline?

In [ ]:
# Build kill-stage attribution for all recordings
import pandas as pd

attr_rows = []
for _name, _tr in all_traces.items():
    _iters = _tr.iterations
    if not _iters:
        continue
    _n0  = _iters[0]['n_candidates']
    _nc  = _iters[-1]['n_candidates']
    _npf = len(_tr.burstlets_pre_gates)
    _npp = _tr.participation_gate['n_post'] if _tr.participation_gate else _npf
    _npm = _tr.bmi_gate['n_pre']  if _tr.bmi_gate else _npp
    _npb = _tr.bmi_gate['n_post'] if _tr.bmi_gate else _npm
    if _tr.gmm and 'kept_event_mask' in _tr.gmm:
        _npg = int(_tr.gmm['kept_event_mask'].sum())
    else:
        _npg = _npb
    for _stage, _surv, _drop in [
        ('iter trim',       _npf,  max(0, _nc - _npf)),
        ('participation',   _npp,  _npf - _npp),
        ('BMI/LLR',         _npb,  _npm - _npb),
        ('GMM',             _npg,  max(0, _npb - _npg)),
    ]:
        attr_rows.append({'recording': _name, 'stage': _stage,
                          'survivors': _surv, 'dropped': _drop})

attr_df = pd.DataFrame(attr_rows)
display(attr_df.pivot_table(index='recording', columns='stage',
                            values=['survivors', 'dropped'], aggfunc='first'))

# ── Plotly grouped bar: survivors vs dropped per stage, faceted by recording ──
_stages_order = ['iter trim', 'participation', 'BMI/LLR', 'GMM']
_names = sorted(attr_df['recording'].unique())

fig = make_subplots(rows=1, cols=len(_names),
                    subplot_titles=_names,
                    shared_yaxes=True)

for _col, _rec in enumerate(_names, 1):
    _sub = attr_df[attr_df['recording'] == _rec].set_index('stage')
    _sub = _sub.reindex(_stages_order)
    fig.add_trace(go.Bar(name='survived', x=_stages_order,
                         y=_sub['survivors'].values,
                         marker_color='steelblue',
                         hovertemplate='%{x}<br>survived=%{y}<br>rec=' + _rec,
                         showlegend=(_col == 1)),
                  row=1, col=_col)
    fig.add_trace(go.Bar(name='dropped', x=_stages_order,
                         y=_sub['dropped'].values,
                         base=_sub['survivors'].values,
                         marker_color='crimson',
                         hovertemplate='%{x}<br>dropped=%{y}<br>rec=' + _rec,
                         showlegend=(_col == 1)),
                  row=1, col=_col)

fig.update_layout(barmode='stack', title='Kill-stage attribution across recordings',
                  height=420, legend_title='count type')
fig.show()
save_html(fig, 'summary_kill_stages.html')


In [ ]:
# Stage 1 — composite signal with iteration slider
# One animated figure per recording (one HTML per recording).

for _name, _tr in all_traces.items():
    _t   = _tr.t_centers
    _iters = _tr.iterations
    if not _iters:
        continue

    _frames = []
    for _k, _snap in enumerate(_iters):
        _comp = _snap['composite']
        _thr  = _snap['composite_threshold']
        _base = _snap['composite_baseline']
        _cands = _snap['candidates']
        _n    = _snap['n_candidates']

        # Candidate windows as shaded regions via a filled scatter trick
        _shade_x, _shade_y = [], []
        for _c in _cands:
            _shade_x += [_c['start'], _c['start'], _c['end'], _c['end'], None]
            _shade_y += [_comp.min() - 0.05, _comp.max() + 0.05,
                         _comp.max() + 0.05, _comp.min() - 0.05, None]

        _frame_data = [
            go.Scatter(x=_t, y=_comp, mode='lines',
                       line=dict(color='steelblue', width=0.8),
                       name='composite',
                       hovertemplate='t=%{x:.2f}s  composite=%{y:.3f}'),
            go.Scatter(x=[_t[0], _t[-1]], y=[_thr, _thr], mode='lines',
                       line=dict(color='red', dash='dash', width=1),
                       name=f'thr={_thr:.2f}', hoverinfo='skip'),
            go.Scatter(x=[_t[0], _t[-1]], y=[_base, _base], mode='lines',
                       line=dict(color='gray', dash='dot', width=0.8),
                       name=f'base={_base:.2f}', hoverinfo='skip'),
            go.Scatter(x=_shade_x, y=_shade_y, fill='toself',
                       fillcolor='rgba(255,165,0,0.12)', line=dict(width=0),
                       name='candidates', hoverinfo='skip'),
        ]
        _frames.append(go.Frame(
            data=_frame_data, name=str(_k),
            layout=go.Layout(
                title_text=(f'{_name} — iter {_k}  '
                            f'n_candidates={_n}  '
                            f"delta={_snap['convergence_delta']:.4f}"),
                xaxis_autorange=True, yaxis_autorange=True,
            ),
        ))

    _slider_steps = [dict(args=[[f.name], dict(mode='immediate',
                                               frame=dict(duration=0))],
                          label=f.name, method='animate')
                     for f in _frames]

    _fig = go.Figure(
        data=_frames[0].data,
        frames=_frames,
        layout=go.Layout(
            title=f'{_name} — composite signal (use slider to step through iterations)',
            xaxis_title='time (s)', yaxis_title='composite score',
            hovermode='x unified',
            updatemenus=[dict(
                type='buttons', showactive=False, y=1.15, x=0.05,
                buttons=[
                    dict(label='▶ Play', method='animate',
                         args=[None, dict(frame=dict(duration=600),
                                          transition=dict(duration=0),
                                          fromcurrent=True)]),
                    dict(label='⏸ Pause', method='animate',
                         args=[[None], dict(mode='immediate',
                                            frame=dict(duration=0))]),
                ],
            )],
            sliders=[dict(
                steps=_slider_steps,
                currentvalue=dict(prefix='iteration: ', font=dict(size=13)),
                pad=dict(t=50),
            )],
            height=500,
        ),
    )
    _fig.show()
    save_html(_fig, f'{_name}_stage1_composite_slider.html')


In [ ]:
# Stage 2 — participation floor (facet scatter, all recordings)
_rows2 = []
for _name, _tr in all_traces.items():
    _gate  = _tr.participation_gate or {}
    _floor = float(_gate.get('floor', 0.0))
    for _ev in _tr.burstlets_pre_gates:
        _rows2.append({
            'recording':     _name,
            'peak_synchrony': float(_ev.get('peak_synchrony', 0)),
            'duration_s':    float(_ev.get('duration_s', 0)),
            'peak_time':     float(_ev.get('peak_time', 0)),
            'kept':          'kept' if float(_ev.get('peak_synchrony', 0)) >= _floor else 'killed',
        })
_df2 = pd.DataFrame(_rows2)

if not _df2.empty:
    fig = px.scatter(
        _df2, x='peak_synchrony', y='duration_s',
        color='peak_time', symbol='kept',
        facet_col='recording', facet_col_wrap=3,
        color_continuous_scale='Viridis',
        symbol_map={'kept': 'circle', 'killed': 'x'},
        hover_data=['recording', 'peak_time', 'peak_synchrony', 'duration_s'],
        labels={'peak_time': 'peak time (s)'},
        title='Stage 2 — Participation floor (color = timestamp, shape = kept/killed)',
        height=420,
    )
    # Add floor vline per facet
    _floor_val = next(
        (float((tr.participation_gate or {}).get('floor', 0))
         for tr in all_traces.values() if tr.participation_gate), 0.0
    )
    fig.add_vline(x=_floor_val, line_dash='dash', line_color='red',
                  annotation_text=f'floor={_floor_val:.3f}',
                  annotation_position='top right')
    fig.show()
    save_html(fig, 'summary_stage2_participation.html')
else:
    print('no pre-floor events found')


In [ ]:
# Stage 3 — BMI / LLR gate (facet scatter, all recordings)
_rows3 = []
for _name, _tr in all_traces.items():
    _bmi = _tr.bmi_gate or {}
    _thr = float(_bmi.get('threshold', 0.0))
    for _ev in _bmi.get('pre_events', []):
        _rows3.append({
            'recording':    _name,
            'llr_aggregate': float(_ev.get('llr_aggregate', 0)),
            'participation': float(_ev.get('participation', 0)),
            'start':        float(_ev.get('start', 0)),
            'kept':         'kept' if float(_ev.get('llr_aggregate', 0)) >= _thr else 'killed',
        })
_df3 = pd.DataFrame(_rows3)

if not _df3.empty:
    fig = px.scatter(
        _df3, x='llr_aggregate', y='participation',
        color='start', symbol='kept',
        facet_col='recording', facet_col_wrap=3,
        color_continuous_scale='Viridis',
        symbol_map={'kept': 'circle', 'killed': 'x'},
        hover_data=['recording', 'start', 'llr_aggregate', 'participation'],
        labels={'start': 'event start (s)'},
        title='Stage 3 — BMI/LLR gate (color = event start time, shape = kept/killed)',
        height=420,
    )
    _thr_val = next(
        (float((tr.bmi_gate or {}).get('threshold', 0))
         for tr in all_traces.values() if tr.bmi_gate), 0.1
    )
    fig.add_vline(x=_thr_val, line_dash='dash', line_color='red',
                  annotation_text=f'min_bmi={_thr_val:.2f}',
                  annotation_position='top right')
    fig.show()
    save_html(fig, 'summary_stage3_bmi.html')
else:
    print('no BMI gate events found')


In [ ]:
# Stage 4 — GMM event clustering PCA (2×2 subplots, one figure per recording)
from sklearn.decomposition import PCA as _PCA

for _name, _tr in all_traces.items():
    _gmm = _tr.gmm
    if not _gmm or 'X' not in _gmm:
        print(f'{_name}: GMM skipped — {(_gmm or {}).get("skipped", "no trace")}')
        continue

    _X, _Xs   = _gmm['X'], _gmm['X_scaled']
    _labels   = _gmm['labels']
    _centers  = _gmm['component_means_scaled']
    _mgroups  = _gmm['merged_groups']
    _cscores  = np.asarray(_gmm['cluster_scores'])
    _kept     = _gmm['kept_event_mask']
    _fnames   = _gmm['feature_names']

    _pca = _PCA(n_components=2).fit(_Xs)
    _Z   = _pca.transform(_Xs)
    _Zc  = _pca.transform(_centers)
    _ev  = _pca.explained_variance_ratio_

    _comp_to_grp = {}
    for _gi, _g in enumerate(_mgroups):
        for _m in _g['members']:
            _comp_to_grp[int(_m)] = _gi
    _grp_id = np.array([_comp_to_grp.get(int(lb), -1) for lb in _labels])

    _cscale = [[0, '#2166ac'], [0.5, '#f7f7f7'], [1, '#b2182b']]  # RdBu-like

    _fig = make_subplots(rows=2, cols=2,
                         subplot_titles=[
                             f'(a) Initial GMM components (n={len(_centers)})',
                             f'(b) Merged groups (n={len(_mgroups)})',
                             '(c) Kept vs killed (color=event time)',
                             '(d) Cluster score (blue=burst)',
                         ])

    # (a) initial components
    _cmap_a = px.colors.qualitative.Plotly
    for _k in np.unique(_labels):
        _m = _labels == _k
        _col = _cmap_a[int(_k) % len(_cmap_a)]
        _fig.add_trace(go.Scatter(
            x=_Z[_m, 0], y=_Z[_m, 1], mode='markers',
            marker=dict(color=_col, size=6, opacity=0.7),
            name=f'comp {_k}', legendgroup=f'comp{_k}',
            hovertemplate=f'comp {_k}<br>PC1=%{{x:.2f}}<br>PC2=%{{y:.2f}}'),
            row=1, col=1)
    _fig.add_trace(go.Scatter(x=_Zc[:, 0], y=_Zc[:, 1], mode='markers',
                              marker=dict(symbol='x', size=12, color='black'),
                              name='centroids', showlegend=False,
                              hoverinfo='skip'), row=1, col=1)

    # (b) merged groups
    _cmap_b = px.colors.qualitative.Set1
    for _gi, _g in enumerate(_mgroups):
        _m = _grp_id == _gi
        _col = _cmap_b[_gi % len(_cmap_b)]
        _sc = _cscores[_gi] if _gi < len(_cscores) else 0
        _fig.add_trace(go.Scatter(
            x=_Z[_m, 0], y=_Z[_m, 1], mode='markers',
            marker=dict(color=_col, size=6, opacity=0.7),
            name=f'grp{_gi} score={_sc:+.2f}',
            hovertemplate=f'group {_gi}  score={_sc:+.2f}<br>'
                          'PC1=%{x:.2f}<br>PC2=%{y:.2f}'),
            row=1, col=2)

    # (c) kept vs killed — color = event index as proxy for time ordering
    _ev_times = np.arange(len(_Z))  # no event timestamps in GMM dict; use index
    _fig.add_trace(go.Scatter(
        x=_Z[_kept, 0], y=_Z[_kept, 1], mode='markers',
        marker=dict(color=_ev_times[_kept], colorscale='Viridis',
                    symbol='circle', size=6, opacity=0.8,
                    colorbar=dict(title='event index', len=0.45, y=0.75)),
        name='kept',
        hovertemplate='kept<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}'), row=2, col=1)
    _fig.add_trace(go.Scatter(
        x=_Z[~_kept, 0], y=_Z[~_kept, 1], mode='markers',
        marker=dict(color=_ev_times[~_kept], colorscale='Viridis',
                    symbol='x', size=7, opacity=0.8, showscale=False),
        name='killed',
        hovertemplate='killed<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}'), row=2, col=1)

    # (d) cluster score continuous
    _per_ev_score = np.array([_cscores[g] if g >= 0 else np.nan for g in _grp_id])
    _abs_max = float(np.nanmax(np.abs(_per_ev_score))) if np.isfinite(_per_ev_score).any() else 1.0
    _fig.add_trace(go.Scatter(
        x=_Z[:, 0], y=_Z[:, 1], mode='markers',
        marker=dict(color=_per_ev_score, colorscale='RdBu_r',
                    cmin=-_abs_max, cmax=_abs_max, size=6,
                    colorbar=dict(title='score', len=0.45, y=0.25)),
        name='score', showlegend=False,
        hovertemplate='score=%{marker.color:.2f}<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}'),
        row=2, col=2)

    _fig.update_layout(
        title=f'Stage 4 GMM clustering — {_name}  '
              f'(PC1+PC2 = {_ev.sum()*100:.1f}% var)',
        height=700,
    )
    _fig.update_xaxes(title_text=f'PC1 ({_ev[0]*100:.1f}%)')
    _fig.update_yaxes(title_text=f'PC2 ({_ev[1]*100:.1f}%)')
    _fig.show()
    save_html(_fig, f'{_name}_stage4_gmm.html')


In [ ]:
# Cross-stage flow — stacked bars, one per recording
_stages_ord = ['iter trim', 'participation', 'BMI/LLR', 'GMM']
_names_all  = sorted(attr_df['recording'].unique())

fig = go.Figure()
for _stage in _stages_ord:
    _sub = attr_df[attr_df['stage'] == _stage].set_index('recording').reindex(_names_all)
    fig.add_trace(go.Bar(
        name=f'{_stage} survived',
        x=_names_all, y=_sub['survivors'].values,
        marker_color='steelblue',
        hovertemplate='%{x}<br>stage=' + _stage + '<br>survived=%{y}',
    ))
    fig.add_trace(go.Bar(
        name=f'{_stage} dropped',
        x=_names_all, y=_sub['dropped'].values,
        marker_color='crimson',
        hovertemplate='%{x}<br>stage=' + _stage + '<br>dropped=%{y}',
    ))

fig.update_layout(barmode='stack',
                  title='Cross-stage kill flow (per recording)',
                  xaxis_title='recording', yaxis_title='events',
                  height=450)
fig.show()
save_html(fig, 'summary_cross_stage_flow.html')


## Notes / conclusions

_Fill in after inspecting the plots above._

- Which stage drops the most candidates on this recording?
- In the GMM PCA panels, do the killed (crimson) events cluster tightly, or are they smeared along the kept side?
- Which feature(s) push the killed merged group below the `score = 0` line?
- Would lowering `min_burst_modulation`, the GMM `cluster_min_separation`, or the kill score threshold recover the lost candidates?

## Per-bin PCA across LDA iterations — `cx138_44_02` deep dive

The cells above operate on the **event-level** GMM (one row per detected event).
That view is downstream of the LDA refinement loop, so it cannot show why a
particular bin was assigned to the burst class in the first place.

`cx138_44_02` is heterogeneous: about 10 bursts at the end and silence at the
start. With the default config, the BMI/LLR gate deletes every burstlet; with
the gate disabled, the algorithm wraps the silent stretch into one giant
"burst". Both failures point to the **per-bin two-class partition** as the
real culprit.

The next cells:

1. reload the recording as `cx138_44_02`,
2. run the detector twice (default vs `min_burst_modulation=0`),
3. project `X_norm` (the feature matrix the Fisher LDA actually fits) to 2D
   PCA at every iteration so the boundary drift is visible,
4. fit `GaussianMixture` with `k ∈ {2,3,4,5}` on the **converged-iteration**
   bin features to test whether the data really wants more than two classes.


In [ ]:
# A — Set the recording for the LDA deep-dive (SOURCE_NAME from Configuration cell)
assert SOURCE_NAME in all_traces, (
    f"'{SOURCE_NAME}' not found in all_traces. "
    f"Available: {list(all_traces.keys())}"
)
SOURCE = next(s for s in sources if SOURCE_NAME in str(s))
RECORDING_NAME = SOURCE.name if SOURCE.is_dir() else SOURCE.parent.name
spike_times    = all_spike_times[SOURCE_NAME]
n_units        = len(spike_times)
total_spikes   = sum(s.size for s in spike_times.values())
t_min = min(float(s.min()) for s in spike_times.values() if s.size)
t_max = max(float(s.max()) for s in spike_times.values() if s.size)
print(f'recording: {RECORDING_NAME}')
print(f'units: {n_units}  spikes: {total_spikes}  duration: {t_max - t_min:.1f} s')


### B — Two reference runs

`trace_default` = stock config (BMI gate at 0.1, GMM on).
`trace_no_gate` = `min_burst_modulation=0` and `cluster_events=False` — keeps everything and over-merges without the quality gates.

These two traces are used by sections C–G to show the before/after LDA geometry. On `cx138_44_02`, `trace_default` now returns ~10 burstlets after the silence-excision fix; the `trace_no_gate` run still illustrates the over-merging failure mode.

In [ ]:
# B — Run the detector twice with traces attached
config_default = IterativeBurstConfig()
trace_default = IterativeBurstTrace()
result_default = compute_iterative_bursts(
    spike_times, config=config_default, debug=True, trace=trace_default,
)

print()
print('---  no-gate run  ---')
config_no_gate = IterativeBurstConfig(min_burst_modulation=0.0, cluster_events=False)
trace_no_gate = IterativeBurstTrace()
result_no_gate = compute_iterative_bursts(
    spike_times, config=config_no_gate, debug=True, trace=trace_no_gate,
)

print()
print(f'default      : burstlets={len(result_default.burstlets)}  net_bursts={len(result_default.network_bursts)}')
print(f'no-gate run  : burstlets={len(result_no_gate.burstlets)}  net_bursts={len(result_no_gate.network_bursts)}')

print()
print(f'trace_default iterations : {len(trace_default.iterations)}')
print(f'trace_no_gate iterations : {len(trace_no_gate.iterations)}')
print(f'feature_names            : {trace_default.feature_names}')
print(f'X_norm shape (iter 0)    : {trace_default.iterations[0]["X_norm"].shape}')


### C — Per-iteration 2D PCA grid

For each shown iteration, fit `PCA(n_components=2)` on that iteration's
`X_norm` and scatter all bins colored by the burst label **output by that
iteration's LDA**. The orange arrow is the Fisher direction `w` projected into
PC space — it points along the axis the LDA is using to separate classes.

Per-iteration PCA fit is intentional: the z-norm changes each iteration, so
a fixed PCA basis would not reflect what the LDA actually saw at step k.

Flip `TRACE_TO_PLOT` to inspect the other run, and set `PLOT_ALL_ITERS=True`
for the full per-iteration sweep.


In [ ]:
# C — Per-iteration LDA PCA with iteration slider (Plotly)
from sklearn.decomposition import PCA as _PCA

_trace_map = {'default': trace_default, 'no_gate': trace_no_gate}
_trace_obj = _trace_map.get(TRACE_TO_PLOT, trace_default)
_iters     = _trace_obj.iterations
_t_centers = _trace_obj.t_centers
_fnames    = _trace_obj.feature_names

if PLOT_ALL_ITERS:
    _show_iters = list(range(len(_iters)))
else:
    _show_iters = sorted({0, len(_iters) // 2, len(_iters) - 1})

# Build one frame per iteration (only shown iters to keep file size manageable)
_frames = []
for _k in _show_iters:
    _snap  = _iters[_k]
    _Xn    = _snap['X_norm']
    _mask  = _snap['candidate_mask'].astype(bool)
    _w     = _snap['w']
    _pca   = _PCA(n_components=2).fit(_Xn)
    _Z     = _pca.transform(_Xn)
    _ev    = _pca.explained_variance_ratio_
    # Fisher arrow endpoint in PC space
    _w_pc  = _pca.components_ @ _w
    _scale = float(np.abs(_Z).max()) * 0.5 / max(float(np.linalg.norm(_w_pc)), 1e-9)

    # Per-bin custom data: [time, PFR, P, FF0, FF1, FF2, FF3, LLR, burstiness]
    _cdata_bg  = np.column_stack([_t_centers[~_mask], _Xn[~_mask]])
    _cdata_bst = np.column_stack([_t_centers[_mask],  _Xn[_mask]])
    _htmpl = ('t=%{customdata[0]:.2f}s<br>'
              'PFR=%{customdata[1]:.1f}<br>'
              'P=%{customdata[2]:.2f}<br>'
              'LLR=%{customdata[7]:.2f}<extra></extra>')

    _top3 = sorted(zip(_fnames, _w), key=lambda x: -abs(x[1]))[:3]
    _wstr = '  '.join(f'{n}={v:+.2f}' for n, v in _top3)

    _frames.append(go.Frame(
        data=[
            go.Scatter(x=_Z[~_mask, 0], y=_Z[~_mask, 1], mode='markers',
                       marker=dict(color=_t_centers[~_mask], colorscale='Viridis',
                                   symbol='circle', size=3, opacity=0.35,
                                   colorbar=dict(title='time (s)', len=0.6)),
                       name='background', customdata=_cdata_bg,
                       hovertemplate=_htmpl),
            go.Scatter(x=_Z[_mask, 0], y=_Z[_mask, 1], mode='markers',
                       marker=dict(color=_t_centers[_mask], colorscale='Viridis',
                                   symbol='diamond', size=5, opacity=0.7,
                                   showscale=False),
                       name='burst', customdata=_cdata_bst,
                       hovertemplate=_htmpl),
            go.Scatter(x=[0, _w_pc[0]*_scale], y=[0, _w_pc[1]*_scale],
                       mode='lines+markers',
                       line=dict(color='darkorange', width=2),
                       marker=dict(symbol='arrow', size=10, color='darkorange',
                                   angleref='previous'),
                       name='Fisher w', hoverinfo='skip'),
        ],
        name=str(_k),
        layout=go.Layout(
            title_text=(f'{RECORDING_NAME} — iter {_k}  '
                        f"n_cand={_snap['n_candidates']}  "
                        f"thr={_snap['composite_threshold']:.2f}  "
                        f"Δ={_snap['convergence_delta']:.4f}<br>"
                        f'top w: {_wstr}  '
                        f'PC1={_ev[0]*100:.1f}%  PC2={_ev[1]*100:.1f}%'),
            xaxis_autorange=True, yaxis_autorange=True,
        ),
    ))

_slider_steps = [
    dict(args=[[f.name], dict(mode='immediate', frame=dict(duration=0))],
         label=f'iter {f.name}', method='animate')
    for f in _frames
]

fig_c = go.Figure(
    data=_frames[0].data,
    frames=_frames,
    layout=go.Layout(
        title=f'{RECORDING_NAME} — LDA PCA per iteration (TRACE={TRACE_TO_PLOT})',
        xaxis_title='PC1', yaxis_title='PC2',
        hovermode='closest',
        updatemenus=[dict(
            type='buttons', showactive=False, y=1.1, x=0.05,
            buttons=[
                dict(label='▶', method='animate',
                     args=[None, dict(frame=dict(duration=700),
                                      transition=dict(duration=0),
                                      fromcurrent=True)]),
                dict(label='⏸', method='animate',
                     args=[[None], dict(mode='immediate', frame=dict(duration=0))]),
            ],
        )],
        sliders=[dict(steps=_slider_steps,
                      currentvalue=dict(prefix='iteration: ', font=dict(size=12)),
                      pad=dict(t=50))],
        height=580,
    ),
)
fig_c.show()
save_html(fig_c, f'{SOURCE_NAME}_sectionC_lda_pca.html')


### D — Boundary shift per iteration (input vs output labels)

Same PCA basis as section C, side by side: left = bins **fed to** this
iteration's Fisher fit (`candidate_mask_in`); right = bins **labelled by**
this iteration's composite (`candidate_mask`). The differences expose how
aggressively each iteration moves the boundary.


In [ ]:
# D — Boundary shift per iteration (Plotly: two-column facet, input vs output)
from sklearn.decomposition import PCA as _PCA

_iters_d   = _trace_obj.iterations
_t_centers = _trace_obj.t_centers
_show_io   = sorted({0, len(_iters_d) // 2, len(_iters_d) - 1})

fig_d = make_subplots(
    rows=len(_show_io), cols=2,
    column_titles=['Input labels', 'Output labels'],
    row_titles=[f'iter {k}' for k in _show_io],
    shared_xaxes=False, shared_yaxes=False,
)

_HTMPL = 't=%{customdata:.2f}s<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}<extra></extra>'

for _row, _k in enumerate(_show_io, 1):
    _snap = _iters_d[_k]
    _Xn   = _snap['X_norm']
    _pca  = _PCA(n_components=2).fit(_Xn)
    _Z    = _pca.transform(_Xn)
    for _col, (_mask, _lbl) in enumerate([
        (_snap['candidate_mask_in'].astype(bool), 'input'),
        (_snap['candidate_mask'].astype(bool),    'output'),
    ], 1):
        _show_lg = (_row == 1 and _col == 1)
        fig_d.add_trace(go.Scatter(
            x=_Z[~_mask, 0], y=_Z[~_mask, 1], mode='markers',
            marker=dict(color=_t_centers[~_mask], colorscale='Viridis',
                        symbol='circle', size=3, opacity=0.3, showscale=(_col==1 and _row==1),
                        colorbar=dict(title='time (s)', len=0.3, y=0.85)),
            name='background', legendgroup='bg', showlegend=_show_lg,
            customdata=_t_centers[~_mask], hovertemplate=_HTMPL,
        ), row=_row, col=_col)
        fig_d.add_trace(go.Scatter(
            x=_Z[_mask, 0], y=_Z[_mask, 1], mode='markers',
            marker=dict(color=_t_centers[_mask], colorscale='Viridis',
                        symbol='diamond', size=5, opacity=0.7, showscale=False),
            name='burst', legendgroup='burst', showlegend=_show_lg,
            customdata=_t_centers[_mask], hovertemplate=_HTMPL,
        ), row=_row, col=_col)

fig_d.update_layout(
    title=f'{RECORDING_NAME} — iteration boundary shift (TRACE={TRACE_TO_PLOT})',
    height=380 * len(_show_io),
)
fig_d.show()
save_html(fig_d, f'{SOURCE_NAME}_sectionD_boundary_shift.html')


### E — 3D PCA at the converged iteration

Sanity check: in 3D, do the burst and background bins form two clean blobs,
or is there visible substructure (e.g. silence vs random firing) that the
LDA is folding into a single "background" class?


In [ ]:
# E — 3D PCA at the converged iteration (Plotly Scatter3d)
from sklearn.decomposition import PCA as _PCA

_snap_f = _trace_obj.iterations[-1]
_Xn     = _snap_f['X_norm']
_labels = _snap_f['candidate_mask'].astype(bool)
_t_c    = _trace_obj.t_centers

_pca3 = _PCA(n_components=3).fit(_Xn)
_Z3   = _pca3.transform(_Xn)
_ev3  = _pca3.explained_variance_ratio_

_HTMPL3 = 't=%{customdata:.2f}s<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}<br>PC3=%{z:.2f}<extra></extra>'

fig_e = go.Figure()
fig_e.add_trace(go.Scatter3d(
    x=_Z3[~_labels, 0], y=_Z3[~_labels, 1], z=_Z3[~_labels, 2],
    mode='markers',
    marker=dict(color=_t_c[~_labels], colorscale='Viridis', size=2, opacity=0.3,
                colorbar=dict(title='time (s)', len=0.6, x=1.05)),
    name='background', customdata=_t_c[~_labels], hovertemplate=_HTMPL3,
))
fig_e.add_trace(go.Scatter3d(
    x=_Z3[_labels, 0], y=_Z3[_labels, 1], z=_Z3[_labels, 2],
    mode='markers',
    marker=dict(color=_t_c[_labels], colorscale='Viridis',
                symbol='diamond', size=4, opacity=0.8, showscale=False),
    name='burst', customdata=_t_c[_labels], hovertemplate=_HTMPL3,
))
fig_e.update_layout(
    title=f'{RECORDING_NAME} — 3D PCA (converged iteration, TRACE={TRACE_TO_PLOT})',
    scene=dict(
        xaxis_title=f'PC1 ({_ev3[0]*100:.1f}%)',
        yaxis_title=f'PC2 ({_ev3[1]*100:.1f}%)',
        zaxis_title=f'PC3 ({_ev3[2]*100:.1f}%)',
    ),
    height=620,
)
fig_e.show()
save_html(fig_e, f'{SOURCE_NAME}_sectionE_3d_pca.html')


### F — Multi-k GMM on bin features

On the **converged-iteration** `X_norm`, fit `GaussianMixture` with
`n_components ∈ {2, 3, 4, 5}`. BIC / AIC for each k are printed; the
panels project every fit into the same converged-PCA basis.

On `cx138_44_02` (silence + tonic + burst) BIC prefers k ≥ 3, confirming
the three-regime structure. The silence-excision fix in the detector
addresses this without needing multi-class LDA — silent bins are simply
excluded from the Fisher fit.

In [ ]:
# F — Multi-k GMM BIC sweep on converged bin features (Plotly facet scatter)
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA as _PCA

_snap_f  = _trace_obj.iterations[-1]
_Xn_f    = _snap_f['X_norm']
_fnames  = _trace_obj.feature_names
_t_c_f   = _trace_obj.t_centers

_pca2_f  = _PCA(n_components=2).fit(_Xn_f)
_Z2_f    = _pca2_f.transform(_Xn_f)
_ev2_f   = _pca2_f.explained_variance_ratio_

_ks      = [2, 3, 4, 5]
_gmm_fits = {}
_bic_rows = []
for _k in _ks:
    _gm = GaussianMixture(n_components=_k, n_init=5, random_state=42, reg_covar=1e-6).fit(_Xn_f)
    _cl = _gm.predict(_Xn_f)
    _bic_rows.append({'k': _k, 'BIC': float(_gm.bic(_Xn_f)), 'AIC': float(_gm.aic(_Xn_f))})
    _gmm_fits[_k] = (_gm, _cl)

bic_df = pd.DataFrame(_bic_rows).set_index('k')
print('BIC / AIC by k (lower is better):')
display(bic_df.round(1))
best_k = int(bic_df['BIC'].idxmin())
print(f'\nBIC-preferred k = {best_k}')

# 1×4 facet scatter: one panel per k, color=timestamp (Viridis)
fig_f = make_subplots(rows=1, cols=len(_ks),
                      subplot_titles=[f'k={k}  BIC={bic_df.loc[k,"BIC"]:.0f}' for k in _ks])
_cmap_f = px.colors.qualitative.Plotly

for _ci, _k in enumerate(_ks, 1):
    _gm, _cl = _gmm_fits[_k]
    for _c in range(_k):
        _m = _cl == _c
        fig_f.add_trace(go.Scatter(
            x=_Z2_f[_m, 0], y=_Z2_f[_m, 1], mode='markers',
            marker=dict(color=_t_c_f[_m], colorscale='Viridis',
                        size=3, opacity=0.55, showscale=(_ci==1 and _c==0),
                        colorbar=dict(title='time (s)', len=0.6)),
            name=f'k={_k} c{_c}',
            customdata=_t_c_f[_m],
            hovertemplate='t=%{customdata:.2f}s<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}<extra></extra>',
        ), row=1, col=_ci)

fig_f.update_layout(
    title=f'{RECORDING_NAME} — Bin-level GMM k-sweep on converged X_norm (TRACE={TRACE_TO_PLOT})',
    height=420, showlegend=False,
)
fig_f.update_xaxes(title_text=f'PC1 ({_ev2_f[0]*100:.1f}%)')
fig_f.update_yaxes(title_text=f'PC2 ({_ev2_f[1]*100:.1f}%)', col=1)
fig_f.show()
save_html(fig_f, f'{SOURCE_NAME}_sectionF_gmm_bic_sweep.html')

# Centroids in standardised feature space
print(f'\nCluster centroids (k={best_k}):')
_gm_best, _cl_best = _gmm_fits[best_k]
_centroids_df = pd.DataFrame(_gm_best.means_, columns=_fnames)
_centroids_df.insert(0, 'time_share', [float((_cl_best == c).mean()) for c in range(best_k)])
display(_centroids_df.round(2))


### G — Cluster assignment as a time strip

Project the BIC-preferred k assignment back onto the recording timeline. The
top strip is the per-bin cluster id; below it are the participation and
composite signals so you can visually verify whether a cluster e.g. covers
the late-recording bursts, the silent stretch, or something in between.


In [ ]:
# G — Cluster assignment as a time strip (linked subplots, range slider)
_snap_g   = _trace_obj.iterations[-1]
_t_g      = _trace_obj.t_centers
_comp_g   = _snap_g['composite']
_mask_g   = _snap_g['candidate_mask'].astype(bool)
_thr_g    = _snap_g['composite_threshold']

_result_map = {'default': result_default, 'no_gate': result_no_gate}
_part_g = _result_map.get(TRACE_TO_PLOT, result_default).plot_data['participation_signal']

_gm_best, _cl_best = _gmm_fits[best_k]

fig_g = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.18, 0.36, 0.46],
    subplot_titles=[f'GMM cluster assignment (k={best_k})',
                    'Participation signal',
                    'Composite signal + LDA burst'],
    vertical_spacing=0.06,
)

# Row 1: cluster assignment as a colour strip (heatmap)
fig_g.add_trace(go.Heatmap(
    x=_t_g, y=[0], z=[_cl_best.astype(float)],
    colorscale='Plotly3', showscale=True,
    colorbar=dict(title='cluster', len=0.18, y=0.93, tickvals=list(range(best_k))),
    hovertemplate='t=%{x:.2f}s  cluster=%{z}<extra></extra>',
), row=1, col=1)

# Row 2: participation
fig_g.add_trace(go.Scatter(
    x=_t_g, y=_part_g, mode='lines',
    line=dict(color='steelblue', width=0.8),
    name='participation',
    hovertemplate='t=%{x:.2f}s  P=%{y:.3f}<extra></extra>',
), row=2, col=1)

# Row 3: composite + threshold + LDA burst region
fig_g.add_trace(go.Scatter(
    x=_t_g, y=_comp_g, mode='lines',
    line=dict(color='black', width=0.8),
    name='composite',
    hovertemplate='t=%{x:.2f}s  composite=%{y:.3f}<extra></extra>',
), row=3, col=1)
fig_g.add_hline(y=_thr_g, line_dash='dash', line_color='red',
                annotation_text=f'thr={_thr_g:.2f}',
                annotation_position='bottom right', row=3, col=1)

# Shade burst windows
for _c in _snap_g.get('candidates', []):
    fig_g.add_vrect(x0=_c['start'], x1=_c['end'],
                    fillcolor='crimson', opacity=0.08,
                    layer='below', line_width=0,
                    row='all', col=1)

fig_g.update_layout(
    title=f'{RECORDING_NAME} — GMM time strip  (TRACE={TRACE_TO_PLOT})',
    hovermode='x unified',
    height=560,
    xaxis3=dict(
        rangeslider=dict(visible=True, thickness=0.06),
        title='time (s)',
    ),
)
fig_g.show()
save_html(fig_g, f'{SOURCE_NAME}_sectionG_timestrip.html')


### Notes — what these plots show

- **Section C / D (per-iter PCA):** the LDA burst class (red) should occupy a
  coherent high-PFR / high-P region. After the silence-excision fix, the silent
  stretch no longer contaminates the bg-class mean, so the Fisher direction
  stays positive on PFR / P / LLR throughout all iterations.
- **Section E (3D):** the background bins on `cx138_44_02` contain two
  sub-blobs (silence + tonic firing) that are now handled by excluding silence
  from the LDA fit rather than by a multi-class model.
- **Section F (BIC):** k ≥ 3 is still BIC-preferred — useful to confirm the
  three-regime structure is real, but the detector no longer needs to model it
  explicitly.
- **Section G (time strip):** confirms the BIC-preferred cluster map aligns
  with the recording timeline (silence early, bursts late).